Install & Import

In [ ]:
!pip install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

Hyperparameters

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 128
noise_dim = 100
lr = 0.0002
epochs = 10

Load Dataset (MNIST)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 43.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.23MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 11.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.3MB/s]


Define Generator

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

Define Discriminator, We do NOT apply sigmoid here. We'll use BCEWithLogitsLoss.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.model(x)

Initialize Models

In [ ]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer_G = optim.Adam(generator.parameters(), lr=lr)
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr)

Train the Discriminator (Core Part)

In [ ]:
for epoch in range(epochs):
    for real_images, _ in dataloader:

        real_images = real_images.view(-1, 784).to(device)
        batch_size = real_images.size(0)

        # ----------------------
        # Train Discriminator
        # ----------------------

        # Real labels = 1
        real_labels = torch.ones(batch_size, 1).to(device)

        # Fake labels = 0
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # Forward real data
        real_output = discriminator(real_images)
        loss_real = criterion(real_output, real_labels)

        # Generate fake data
        noise = torch.randn(batch_size, noise_dim).to(device)
        fake_images = generator(noise).detach()  # IMPORTANT: detach

        # Forward fake data
        fake_output = discriminator(fake_images)
        loss_fake = criterion(fake_output, fake_labels)

        # Total discriminator loss
        d_loss = loss_real + loss_fake

        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # ----------------------
        # Train Generator
        # ----------------------

        noise = torch.randn(batch_size, noise_dim).to(device)
        fake_images = generator(noise)

        output = discriminator(fake_images)
        g_loss = criterion(output, real_labels)  # wants D to think fake is real

        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()

    print(f"Epoch [{epoch+1}/{epochs}]  D Loss: {d_loss.item():.4f}  G Loss: {g_loss.item():.4f}")

Epoch [1/10]  D Loss: 0.0897  G Loss: 5.0055
Epoch [2/10]  D Loss: 0.1865  G Loss: 6.6831
Epoch [3/10]  D Loss: 0.6295  G Loss: 2.6589
Epoch [4/10]  D Loss: 0.5716  G Loss: 2.8501
Epoch [5/10]  D Loss: 0.8837  G Loss: 2.0668
Epoch [6/10]  D Loss: 0.1536  G Loss: 3.1575
Epoch [7/10]  D Loss: 0.5069  G Loss: 2.6178
Epoch [8/10]  D Loss: 0.2258  G Loss: 4.1674
Epoch [9/10]  D Loss: 0.2640  G Loss: 4.2923
Epoch [10/10]  D Loss: 0.1777  G Loss: 4.1704


What’s Happening in Discriminator Training?

Real images → label = 1

Fake images → label = 0

Compute BCE loss

Backprop only through discriminator

Update discriminator weights

detach() ensures:

Generator is NOT updated during discriminator training.